# 🏠 Aula 2 — Exercícios para Casa
**Pesquisa Operacional e Otimização em IA** — MBA em Ciência de Dados (UNIFOR)

Prof. Mafra | mafra@verboo.ai

---

Três problemas de Programação Inteira para resolver com PuLP. Use o notebook de demonstração da aula como referência.

### Bibliotecas recomendadas

| Biblioteca | Uso | Instalação |
|---|---|---|
| **PuLP** | Programação Linear e Inteira | `pip install pulp` |
| **OR-Tools** | Otimização combinatória (Google) | `pip install ortools` |

### Documentação

- PuLP: https://coin-or.github.io/pulp/
- OR-Tools: https://developers.google.com/optimization
- Tutorial PuLP (Real Python): https://realpython.com/linear-programming-python/

In [2]:
!pip install pulp -q
from pulp import *


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


---
## Exercício 1 — Seleção de Projetos 💼

Uma empresa de tecnologia tem **10 projetos** candidatos para o próximo trimestre. Cada projeto tem um custo de investimento, um prazo estimado e um retorno esperado (score de 1 a 10 definido pelo comitê).

Os projetos são: **Migração Cloud** (R$ 180k, 12 semanas, score 8), **App Clientes** (R$ 120k, 8 semanas, score 7), **Automação de Testes** (R$ 50k, 4 semanas, score 5), **Data Lake** (R$ 200k, 16 semanas, score 9), **Chatbot de Suporte** (R$ 70k, 6 semanas, score 6), **Redesign do Site** (R$ 40k, 3 semanas, score 4), **API para Partners** (R$ 90k, 8 semanas, score 7), **Dashboard BI** (R$ 60k, 5 semanas, score 5), **Security Audit** (R$ 30k, 2 semanas, score 3) e **ML para Churn** (R$ 150k, 10 semanas, score 8).

O budget disponível é de **R$ 500 mil**. Além disso, o **Data Lake** e a **Migração Cloud** usam a mesma equipe de infraestrutura, então **não podem ser executados ao mesmo tempo** — no máximo um dos dois.

**Objetivo:** maximizar o retorno total dos projetos selecionados.

### O que fazer

1. Defina as variáveis binárias (cada projeto entra ou não)
2. Modele a função objetivo, a restrição de budget e a restrição de exclusividade
3. Resolva e analise: quais projetos ficaram de fora? Por quê?
4. **Extra:** faça análise de sensibilidade variando o budget de R$ 200k a R$ 600k

### Variáveis de Decisão

| Projeto | Custo (R$k) | Prazo (semanas) | Score |
|---------|------------|-----------------|-------|
| Migração Cloud | 180 | 12 | 8 |
| App Clientes | 120 | 8 | 7 |
| Automação de Testes | 50 | 4 | 5 |
| Data Lake | 200 | 16 | 9 |
| Chatbot de Suporte | 70 | 6 | 6 |
| Redesign do Site | 40 | 3 | 4 |
| API para Partners | 90 | 8 | 7 |
| Dashboard BI | 60 | 5 | 5 |
| Security Audit | 30 | 2 | 3 |
| ML para Churn | 150 | 10 | 8 |

**Objetivo: maximizar o score total dos projetos selecionados**


### Restrições

1. **Budget:** o custo total dos projetos selecionados não pode ultrapassar R$ 500k
2. **Exclusividade Data Lake / Migração Cloud:** no máximo um dos dois pode ser selecionado


In [6]:
# Exercício 1 — escreva seu código aqui

projetos = [
    ("Migração Cloud",     180, 12, 8),
    ("App Clientes",       120,  8, 7),
    ("Automação de Testes", 50,  4, 5),
    ("Data Lake",          200, 16, 9),
    ("Chatbot de Suporte",  70,  6, 6),
    ("Redesign do Site",    40,  3, 4),
    ("API para Partners",   90,  8, 7),
    ("Dashboard BI",        60,  5, 5),
    ("Security Audit",      30,  2, 3),
    ("ML para Churn",      150, 10, 8),
]

def resolver(budget):
    prob = LpProblem("Selecao_Projetos", LpMaximize)
    x = [LpVariable(f"proj_{i}", cat="Binary") for i in range(10)]
    prob += lpSum(projetos[i][3] * x[i] for i in range(10)), "Retorno"
    prob += lpSum(projetos[i][1] * x[i] for i in range(10)) <= budget, "Budget"
    prob += x[0] + x[3] <= 1, "Exclusao_DataLake_Migracao"
    prob.solve(PULP_CBC_CMD(msg=0))
    return x, prob

# --- Resolução principal (budget = 500k) ---
x, prob = resolver(500)

print("=" * 55)
print("  SELEÇÃO DE PROJETOS — Budget R$ 500k")
print("=" * 55)
budget_usado = 0
for i, p in enumerate(projetos):
    status = "✅ FAZER" if x[i].varValue == 1 else "   —"
    print(f"  {status}  {p[0]:<20} R${p[1]:>4}k  score {p[3]}")
    if x[i].varValue == 1:
        budget_usado += p[1]
print("=" * 55)
print(f"  Retorno total: {value(prob.objective):.0f}")
print(f"  Budget usado: R$ {budget_usado}k / 500k")

# --- Análise: por que ficaram de fora? ---
print("\n--- Projetos excluídos ---")
for i, p in enumerate(projetos):
    if x[i].varValue == 0:
        motivo = "exclusividade (Data Lake/Migração Cloud)" if i in (0, 3) else "budget insuficiente"
        print(f"  {p[0]:<20} R${p[1]:>4}k  score {p[3]}  → {motivo}")

# --- Extra: análise de sensibilidade ---
print("\n--- Análise de Sensibilidade ---")
print(f"{'Budget':>10}  {'Score':>6}  {'Projetos selecionados'}")
print("-" * 60)
for b in range(200, 650, 50):
    xi, pi = resolver(b)
    selecionados = [projetos[i][0] for i in range(10) if xi[i].varValue == 1]
    print(f"  R${b:>4}k  {value(pi.objective):>6.0f}  {selecionados}")

  SELEÇÃO DE PROJETOS — Budget R$ 500k
     —  Migração Cloud       R$ 180k  score 8
     —  App Clientes         R$ 120k  score 7
  ✅ FAZER  Automação de Testes  R$  50k  score 5
     —  Data Lake            R$ 200k  score 9
  ✅ FAZER  Chatbot de Suporte   R$  70k  score 6
  ✅ FAZER  Redesign do Site     R$  40k  score 4
  ✅ FAZER  API para Partners    R$  90k  score 7
  ✅ FAZER  Dashboard BI         R$  60k  score 5
  ✅ FAZER  Security Audit       R$  30k  score 3
  ✅ FAZER  ML para Churn        R$ 150k  score 8
  Retorno total: 38
  Budget usado: R$ 490k / 500k

--- Projetos excluídos ---
  Migração Cloud       R$ 180k  score 8  → exclusividade (Data Lake/Migração Cloud)
  App Clientes         R$ 120k  score 7  → budget insuficiente
  Data Lake            R$ 200k  score 9  → exclusividade (Data Lake/Migração Cloud)

--- Análise de Sensibilidade ---
    Budget   Score  Projetos selecionados
------------------------------------------------------------
  R$ 200k      18  ['Automação de

---
## Exercício 2 — Escala de Atendimento em Clínica 🏥

Uma clínica médica funciona de segunda a sábado e precisa montar a escala semanal de **6 médicos** em **2 turnos** (manhã e tarde). Cada médico tem um custo diferente por turno trabalhado.

As regras da clínica são: cada turno precisa de pelo menos **2 médicos** presentes. Nenhum médico pode trabalhar mais de **8 turnos** na semana (para evitar burnout). O **Dr. Silva** não pode trabalhar aos sábados (restrição pessoal). A **Dra. Costa** e o **Dr. Lima** não podem trabalhar no mesmo turno (conflito de agenda com outra clínica).

Os custos por turno são: Dr. Silva R$ 800, Dra. Costa R$ 900, Dr. Lima R$ 750, Dra. Santos R$ 850, Dr. Oliveira R$ 700 e Dra. Pereira R$ 950.

**Objetivo:** montar a escala que minimize o custo total da semana, respeitando todas as restrições.

### O que fazer

1. Defina as variáveis binárias: médico `i` trabalha no turno `t` do dia `d`?
2. Quantas variáveis o modelo tem? (6 médicos × 2 turnos × 6 dias = ?)
3. Modele TODAS as restrições (mínimo por turno, máximo por médico, Silva sem sábado, Costa/Lima não juntos)
4. Atenção: aqui o objetivo é **minimizar** (`LpMinimize`)

### Dica

Para criar variáveis em lote:
```python
x = LpVariable.dicts("turno", 
    (medicos, turnos, dias), 
    cat="Binary")
# Acesso: x["Silva"]["manha"]["seg"]
```

### Variáveis de Decisão

| Médico | Custo/turno (R$) |
|--------|-----------------|
| Dr. Silva | 800 |
| Dra. Costa | 900 |
| Dr. Lima | 750 |
| Dra. Santos | 850 |
| Dr. Oliveira | 700 |
| Dra. Pereira | 950 |

**Objetivo: minimizar o custo total da escala semanal**


### Restrições

1. **Mínimo por turno:** cada turno de cada dia precisa de pelo menos 2 médicos
2. **Máximo por médico:** nenhum médico pode trabalhar mais de 8 turnos na semana
3. **Silva sem sábado:** Dr. Silva não trabalha no sábado
4. **Costa e Lima não juntos:** não podem estar no mesmo turno do mesmo dia

In [11]:
# Exercício 2 — escreva seu código aqui

medicos = ["Silva", "Costa", "Lima", "Santos", "Oliveira", "Pereira"]
custo = {"Silva": 800, "Costa": 900, "Lima": 750, "Santos": 850, "Oliveira": 700, "Pereira": 950}
turnos_cli = ["manha", "tarde"]
dias = ["seg", "ter", "qua", "qui", "sex", "sab"]

x = LpVariable.dicts("escala", (medicos, turnos_cli, dias), cat="Binary")
prob2 = LpProblem("Escala_Medica", LpMinimize)

#funcao objetivo
prob2 += lpSum(custo[m] * x[m][t][d] for m in medicos for t in turnos_cli for d in dias), "Custo_Total"


#restricoes
for d in dias:
    for t in turnos_cli:
        prob2 += lpSum(x[m][t][d] for m in medicos) >= 2 # pelo menos 2 médicos por turno

for m in medicos:
    prob2 += lpSum(x[m][t][d] for t in turnos_cli for d in dias) <= 8 # no máximo 8 turnos por médico

# Silva nao trabalha no sabado
prob2 += x["Silva"]["manha"]["sab"] == 0, "Silva_sem_sabado_manha"
prob2 += x["Silva"]["tarde"]["sab"] == 0, "Silva_sem_sabado_tarde"

# Costa e Lima nao podem trabalhar no mesmo turno do mesmo dia
for t in turnos_cli:
    for d in dias:
        prob2 += x["Costa"][t][d] + x["Lima"][t][d] <= 1, f"Costa_Lima_{t}_{d}"

prob2.solve(PULP_CBC_CMD(msg=0))

print(f"Status: {LpStatus[prob2.status]}")
print(f"Custo total: R$ {value(prob2.objective):.0f}")

for d in dias:
    print(f"\nDia {d}:")
    for t in turnos_cli:
        alocados = [m for m in medicos if value(x[m][t][d]) == 1]
        print(f"  {t}: {alocados}")


Status: Optimal
Custo total: R$ 18000

Dia seg:
  manha: ['Silva', 'Lima']
  tarde: ['Lima', 'Oliveira']

Dia ter:
  manha: ['Silva', 'Oliveira']
  tarde: ['Silva', 'Lima']

Dia qua:
  manha: ['Silva', 'Lima']
  tarde: ['Silva', 'Oliveira']

Dia qui:
  manha: ['Lima', 'Oliveira']
  tarde: ['Silva', 'Oliveira']

Dia sex:
  manha: ['Silva', 'Lima']
  tarde: ['Silva', 'Oliveira']

Dia sab:
  manha: ['Lima', 'Oliveira']
  tarde: ['Lima', 'Oliveira']


---
## Exercício 3 — Rota de Entregas 🚚

Uma empresa de logística precisa fazer entregas em **7 cidades** do interior do Ceará e voltar à base em Fortaleza. O caminhão sai de Fortaleza, visita todas as cidades exatamente uma vez e retorna.

As cidades são: **Fortaleza** (base), **Sobral** (230 km), **Juazeiro do Norte** (530 km), **Crato** (540 km), **Quixadá** (170 km), **Itapipoca** (130 km), **Canindé** (110 km) e **Iguatu** (380 km).

Você precisará montar a matriz de distâncias entre todas as cidades (pesquise ou estime). O objetivo é encontrar a rota que **minimize a distância total** percorrida.

### O que fazer

1. Monte a matriz de distâncias (pode usar valores aproximados)
2. Modele como TSP (Caixeiro Viajante) usando variáveis binárias
3. Não esqueça das restrições MTZ para eliminar sub-rotas
4. Resolva e imprima a rota na ordem
5. **Extra:** e se houvesse 2 caminhões? Como mudaria o modelo?

### Dica

Revise o código do TSP que fizemos em aula com as 6 cidades do Nordeste. A estrutura é a mesma — só muda a matriz de distâncias.

In [12]:
# Exercício 3 — Rota de Entregas (TSP)

cidades = ["Fortaleza", "Sobral", "Juazeiro", "Crato", "Quixadá", "Itapipoca", "Canindé", "Iguatu"]
n = len(cidades)

#                  Ftal  Sobr  Juaz  Crat  Quix  Itap  Cani  Igua
dist = [
    [   0,  230,  530,  540,  170,  130,  110,  380],  # Fortaleza
    [ 230,    0,  620,  630,  310,  200,  230,  530],  # Sobral
    [ 530,  620,    0,   30,  390,  620,  430,  180],  # Juazeiro
    [ 540,  630,   30,    0,  400,  630,  440,  190],  # Crato
    [ 170,  310,  390,  400,    0,  270,   80,  260],  # Quixadá
    [ 130,  200,  620,  630,  270,    0,  200,  460],  # Itapipoca
    [ 110,  230,  430,  440,   80,  200,    0,  300],  # Canindé
    [ 380,  530,  180,  190,  260,  460,  300,    0],  # Iguatu
]

prob3 = LpProblem("TSP_Ceara", LpMinimize)

# x[i][j] = 1 se viajamos da cidade i para a cidade j
x = [[LpVariable(f"x_{i}_{j}", cat="Binary") if i != j else None
      for j in range(n)] for i in range(n)]

# Variáveis auxiliares MTZ para eliminar sub-rotas
u = [LpVariable(f"u_{i}", lowBound=1, upBound=n-1) for i in range(n)]

# Objetivo: minimizar distância total
prob3 += lpSum(dist[i][j] * x[i][j] for i in range(n) for j in range(n) if i != j)

# Cada cidade é visitada exatamente 1 vez (entrada)
for j in range(n):
    prob3 += lpSum(x[i][j] for i in range(n) if i != j) == 1

# Cada cidade é saída exatamente 1 vez
for i in range(n):
    prob3 += lpSum(x[i][j] for j in range(n) if i != j) == 1

# Restrições MTZ (eliminam sub-rotas)
for i in range(1, n):
    for j in range(1, n):
        if i != j:
            prob3 += u[i] - u[j] + n * x[i][j] <= n - 1

prob3.solve(PULP_CBC_CMD(msg=0))

# Reconstruir e imprimir a rota
print("=" * 50)
print("  ROTA ÓTIMA — Entregas no Ceará")
print("=" * 50)
rota = [0]
atual = 0
dist_total = 0
for _ in range(n - 1):
    for j in range(n):
        if j != atual and x[atual][j] is not None and x[atual][j].varValue == 1:
            dist_total += dist[atual][j]
            print(f"  {cidades[atual]:>12} → {cidades[j]:<12} {dist[atual][j]:>5} km")
            rota.append(j)
            atual = j
            break
dist_total += dist[atual][0]
print(f"  {cidades[atual]:>12} → {cidades[0]:<12} {dist[atual][0]:>5} km")
print(f"{'─' * 35}")
print(f"  {'TOTAL':>12}   {dist_total:>5} km")
print("=" * 50)

  ROTA ÓTIMA — Entregas no Ceará
     Fortaleza → Canindé        110 km
       Canindé → Quixadá         80 km
       Quixadá → Iguatu         260 km
        Iguatu → Juazeiro       180 km
      Juazeiro → Crato           30 km
         Crato → Sobral         630 km
        Sobral → Itapipoca      200 km
     Itapipoca → Fortaleza      130 km
───────────────────────────────────
         TOTAL    1620 km


---

# 🧠 Preparação para a Aula 3 — Heurísticas

Na aula 2, vocês resolveram os problemas acima com **métodos exatos** (PuLP/Branch-and-Bound), que garantem a solução ótima.

Na próxima aula, vamos ver que para problemas grandes, o solver pode levar **horas, dias ou nunca terminar**. Nesses casos, usamos **heurísticas** — métodos que encontram soluções **boas** (não necessariamente ótimas) em **tempo viável**.

Os 3 exercícios abaixo pedem que vocês resolvam os **mesmos problemas** da aula 2, mas **sem usar o PuLP**. A ideia é comparar: a heurística chegou perto da solução ótima? Quanto tempo levou?

> **Regra:** Nesses exercícios, vocês **NÃO** podem usar PuLP, OR-Tools ou qualquer solver. Apenas Python puro (loops, listas, dicionários).

---
## Exercício 4 — Seleção de Projetos com Heurística Gulosa 🤑

Volte ao problema do **Exercício 1** (10 projetos, budget de R$ 500k). Agora, em vez de usar o PuLP, resolva com uma **heurística gulosa (greedy)**:

**A ideia:** ordene os projetos por algum critério de "eficiência" e vá adicionando um a um enquanto couber no budget.

### O que fazer

1. Calcule a **eficiência** de cada projeto: `score / custo`
2. Ordene os projetos do mais eficiente ao menos eficiente
3. Percorra a lista ordenada: se o projeto cabe no budget restante, selecione-o
4. Não esqueça da restrição: Data Lake e Migração Cloud não podem ir juntos
5. Compare o resultado com a solução ótima do Exercício 1:
   - Quais projetos a heurística selecionou?
   - O score total ficou igual, pior ou melhor que o PuLP?
6. **Extra:** teste outros critérios de ordenação (só por score, só por custo). Qual funciona melhor?

### Dica
```python
projetos = [
    {"nome": "Migração Cloud", "custo": 180, "score": 8},
    {"nome": "App Clientes", "custo": 120, "score": 7},
    # ... complete
]
projetos.sort(key=lambda p: p["score"] / p["custo"], reverse=True)
```

In [5]:
# Exercício 4 — escreva seu código aqui

projetos = [
    {"nome": "Migração Cloud", "custo": 180, "score": 8},
    {"nome": "App Clientes", "custo": 120, "score": 7},
    {"nome": "Automação de Testes", "custo": 50, "score": 5},
    {"nome": "Data Lake", "custo": 200, "score": 9},
    {"nome": "Chatbot de Suporte", "custo": 70, "score": 6},
    {"nome": "Redesign do Site", "custo": 40, "score": 4},
    {"nome": "API para Partners", "custo": 90, "score": 7},
    {"nome": "Dashboard BI", "custo": 60, "score": 5},
    {"nome": "Security Audit", "custo": 30, "score": 3},
    {"nome": "ML para Churn", "custo": 150, "score": 8},
]

budget = 500

# Continue daqui


---
## Exercício 5 — Rota de Entregas com Vizinho Mais Próximo 🗺️

Volte ao problema do **Exercício 3** (TSP com 8 cidades do Ceará). Agora, resolva com a **heurística do Vizinho Mais Próximo (Nearest Neighbor)**:

**A ideia:** comece em Fortaleza. A cada passo, vá para a cidade **mais próxima** que ainda não foi visitada. Quando todas forem visitadas, volte a Fortaleza.

### O que fazer

1. Use a mesma matriz de distâncias do Exercício 3
2. Implemente o algoritmo do Vizinho Mais Próximo partindo de Fortaleza
3. Calcule a distância total da rota encontrada
4. Compare com a solução ótima do PuLP (Exercício 3):
   - A rota é a mesma? A distância ficou muito maior?
   - Calcule o **gap**: `(heurística - ótimo) / ótimo × 100%`
5. **Extra — Heurística de Melhoria (2-opt):** a rota do vizinho mais próximo pode ter cruzamentos. O **2-opt** tenta melhorar trocando pares de arestas. Implemente:
   - Para cada par de arestas, teste se inverter o trecho entre elas reduz a distância
   - Repita até não conseguir mais melhorar
   - Quanto o 2-opt melhorou em relação ao vizinho mais próximo?

### Dica para o 2-opt
```python
def dois_opt(rota, dist):
    melhorou = True
    while melhorou:
        melhorou = False
        for i in range(1, len(rota) - 2):
            for j in range(i + 1, len(rota) - 1):
                # Testa inverter o trecho rota[i:j+1]
                nova_rota = rota[:i] + rota[i:j+1][::-1] + rota[j+1:]
                if custo_rota(nova_rota, dist) < custo_rota(rota, dist):
                    rota = nova_rota
                    melhorou = True
    return rota
```

In [6]:
# Exercício 5 — escreva seu código aqui

cidades = ["Fortaleza", "Sobral", "Juazeiro", "Crato", "Quixadá", "Itapipoca", "Canindé", "Iguatu"]

# Use a mesma matriz de distâncias do Exercício 3
# dist = [
#     [0, ...],
#     ...
# ]

# Implemente o Vizinho Mais Próximo aqui


---
## Exercício 6 — Escala da Clínica com Heurística Construtiva 🏥

Volte ao problema do **Exercício 2** (6 médicos, 6 dias, 2 turnos). Agora, monte a escala **sem solver**, usando uma heurística construtiva:

**A ideia:** para cada turno de cada dia, escolha os 2 médicos mais baratos **que ainda possam trabalhar** (respeitando o limite de 8 turnos, as restrições do Dr. Silva e o conflito Costa/Lima).

### O que fazer

1. Para cada dia e turno, monte a lista de médicos **disponíveis** (que não violam nenhuma restrição)
2. Ordene por custo (mais barato primeiro)
3. Selecione os 2 primeiros da lista
4. Atualize o contador de turnos de cada médico selecionado
5. Ao final, calcule o custo total da escala
6. Compare com a solução ótima do PuLP (Exercício 2):
   - O custo ficou maior? Quanto?
   - A escala ficou "justa" (turnos bem distribuídos) ou sobrecarregou alguém?

### Reflexão

Na heurística gulosa, o médico mais barato (Dr. Oliveira, R$ 700) tende a ser escolhido sempre. Isso pode ser um problema?

- O que acontece quando ele atinge o limite de 8 turnos?
- A escala final ainda é viável?
- Como você mudaria a heurística para distribuir melhor os turnos?

In [7]:
# Exercício 6 — escreva seu código aqui

medicos = ["Silva", "Costa", "Lima", "Santos", "Oliveira", "Pereira"]
custo = {"Silva": 800, "Costa": 900, "Lima": 750, "Santos": 850, "Oliveira": 700, "Pereira": 950}
turnos_cli = ["manha", "tarde"]
dias = ["seg", "ter", "qua", "qui", "sex", "sab"]

# Continue daqui


---

### Referências úteis

| Recurso | Link |
|---------|------|
| Documentação PuLP | https://coin-or.github.io/pulp/ |
| Tutorial PuLP (Real Python) | https://realpython.com/linear-programming-python/ |
| Exemplos PuLP (GitHub) | https://github.com/coin-or/pulp/tree/master/examples |
| OR-Tools TSP | https://developers.google.com/optimization/routing/tsp |
| OR-Tools Scheduling | https://developers.google.com/optimization/scheduling |

**Entrega:** enviem o notebook (.ipynb) com o código funcionário até a próxima aula.